<div dir="rtl">

# 🔍 04 - Vector Retrievers & RAG Chains (تجارب الاسترجاع والسلاسل المتجهة)

## ما هو هذا الكراس؟
يقدم هذا الكراس تطبيقاً مفاهيمياً ومنظماً لاستكشاف **المُسترجعات المتجهة (Vector Retrievers)** وتقنيات البحث الدلالي باستخدام **FAISS** و **HuggingFace Embeddings** وربطها بسلسلة توليدية بـ **Groq**.

## المحاور الرئيسية التي يتم تناولها:
1. **بناء كائنات المستندات**: إنشاء مستندات مع بيانات وصفية غنية (`metadata`).
2. **التضمين والفهرسة المتجهة**: تحويل النصوص لمتجهات وتخزينها في مستودع `FAISS` محلي.
3. **أنماط الاسترجاع والبحث الدلالي**: استخدام `similarity_search_with_score` و `as_retriever` والاستعلام الجماعي (`batch`).
4. **بناء وتدفق سلسلة RAG بـ LCEL**: ربط الـ Retriever بقالب التوجيه وتوليد إجابات موثقة عبر `ChatGroq`.

</div>

### 1️⃣ تحميل البيئة وتهيئة المستندات المرجعية (Documents Creation)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document

# تحميل متغيرات البيئة
load_dotenv(find_dotenv())

# إنشاء مستندات مع بيانات وصفية غنية (Metadata)
documents = [
    Document(
        page_content="Real Madrid is one of the most successful football clubs in Europe. The club has won numerous UEFA Champions League titles and is known for its strong performances in European competitions.",
        metadata={"source": "real_madrid.txt", "topic": "football_club", "club": "Real Madrid"}
    ),
    Document(
        page_content="Barcelona is one of the biggest football clubs in the world, but its European performances have been disappointing in recent seasons. Over roughly the last ten years, Barcelona has struggled to consistently compete at the highest level in the UEFA Champions League, with several early exits and disappointing knockout-stage results. Because of this, Barcelona can be described as an underperforming or unsuccessful club in European competition during this period, despite its strong domestic history and global reputation.",
        metadata={"source": "barcelona.txt", "topic": "football_club", "club": "Barcelona"}
    ),
    Document(
        page_content="Bayern Munich is a major European football club with a strong record in the UEFA Champions League. The club regularly competes for major European trophies and has maintained a high level of performance against top European teams.",
        metadata={"source": "bayern_munich.txt", "topic": "football_club", "club": "Bayern Munich"}
    ),
    Document(
        page_content="Manchester City has become one of the strongest clubs in European football in the modern era. The club has consistently competed deep into the UEFA Champions League and won its first Champions League title in 2023.",
        metadata={"source": "manchester_city.txt", "topic": "football_club", "club": "Manchester City"}
    ),
    Document(
        page_content="Paris Saint-Germain is a prominent French football club that has invested heavily in building competitive squads. PSG has regularly participated in the UEFA Champions League and reached the final in 2020.",
        metadata={"source": "psg.txt", "topic": "football_club", "club": "Paris Saint-Germain"}
    )
]

print(f"✅ تم تجهيز {len(documents)} مستندات مع الميتاداتا بنجاح.")

### 2️⃣ تهيئة نماذج التضمين (HuggingFace Embeddings) وفهرس FAISS

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

# 1. تهيئة نموذج التضمين المحلي
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. إنشاء مستودع المتجهات FAISS وتغذيتها بالمستندات
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=hf_embeddings
)

# 3. تهيئة نموذج التوليد Groq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2
)

print("✅ تم إنشاء فهرس FAISS وتهيئة نموذج Groq بنجاح.")

### 3️⃣ البحث الدلالي وحساب درجات التشابه (Similarity Search with Scores)

In [ ]:
# تنفيذ البحث الدلالي مع إرجاع درجة التشابه (L2 Distance Score)
query = "Which club struggles in European competitions?"
retrieved_docs = vector_store.similarity_search_with_score(query, k=2)

print(f"🔍 نتائج البحث الدلالي للاستعلام: '{query}'\n")
for doc, score in retrieved_docs:
    print(f"📌 النادي: {doc.metadata['club']} (المصدر: {doc.metadata['source']})")
    print(f"📊 درجة المسافة (Score): {score:.4f}")
    print(f"📝 المحتوى: {doc.page_content}\n")
    print("-" * 70)

### 4️⃣ تحويل المستودع إلى Retriever واستعلام الدفعات (Batch Queries)

In [ ]:
# تحويل مستودع FAISS إلى كائن Retriever قياسي
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# تنفيذ استعلام دفعات متوازي (Batch Retrieval)
batch_queries = ["Champions League winners 2023", "French club in Champions League"]
batch_results = retriever.batch(batch_queries)

print(f"✅ تم استرجاع {len(batch_results)} مجموعات نتائج للاستعلام المزدوج.")
for idx, docs in enumerate(batch_results):
    print(f"\n🔎 استعلام الدفعة {idx+1}: '{batch_queries[idx]}'")
    for d in docs:
        print(f"  - {d.metadata['club']}: {d.page_content[:90]}...")

### 5️⃣ بناء وتثبيت سلسلة RAG الكاملة بـ LCEL

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are a football expert assistant. Use the following context to accurately answer the question.

Question:
{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("human", prompt_template)
])

# بناء السلسلة بواسطة مشغل الربط |
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

# تجربة السلسلة على عدة أسئلة استكشافية
queries = [
    "Which club has struggled in European competitions?",
    "Which club won the Champions League in 2023?",
    "Which club reached the Champions League final in 2020?"
]

for q in queries:
    print(f"\n❓ السؤال: {q}")
    res = rag_chain.invoke(q)
    print(f"🤖 الإجابة:\n{res.content}")
    print("=" * 70)

<div dir="rtl">

## 💡 الخلاصة العملية:
- تم بناء مسار استرجاع دلالي متكامل من المستندات الخام وحفظها في فهرس FAISS.
- تم استخدام `as_retriever` وربط المسترجعات بسلاسل LCEL التوليدية للحصول على إجابات موثقة بالسياق.

</div>